# 01 · Data prep: weekly demand per SKU
**Project:** Inventory planning & demand forecasting for a UK online gift retailer

**Data:** UCI *Online Retail II*, the same real invoice data as my returns project (Project 2): 01-Dec-2009 to 09-Dec-2011.
Run `python scripts/download_data.py` first.

Project 2 analysed this retailer's **returns**. This project plans its **stock**: how much to hold of each product and when to reorder.
The first step is to turn a million invoice lines into one clean number per product per week: **net demand**.

| Step | What happens |
|---|---|
| 1 | Clean the invoice lines with the same rules as Project 2 |
| 2 | Build a calendar of 103 full weeks (Mon–Sun), leaving out the partial weeks at both ends and the partial Dec-2011 |
| 3 | Weekly demand per SKU = units sold − units returned (net of returns), with zero weeks filled in |
| 4 | Choose the planning range: the top 150 regular SKUs by sales value |
| 5 | Describe each SKU: average and variability of demand, price, Christmas uplift |
| 6 | ABC–XYZ classification |
| 7 | Check and save the tables that SQL, Excel and Power BI use |

Python is only used here for data prep; the analysis itself is in SQL, Excel and Power BI.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", "{:,.2f}".format)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

prep_log = []

def log(step, rows_before, rows_after, note=""):
    prep_log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after,
                     "rows_removed": rows_before - rows_after, "note": note})

## 1. Clean the invoice lines (same rules as Project 2)
Short version of the Project 2 cleaning. The reasons for each rule are in that project's cleaning notebook:
- upper-case stock codes, remove exact duplicates and double entries
- keep real products only (codes start with 5 digits); drop postage, fees and bad-debt adjustments (`A` invoices)
- drop zero-price lines (stock write-offs) and the two 80,000-unit order-entry mistakes
- credit notes (invoice starts with `C`) are **returns**: negative quantities that we net off demand

In [2]:
raw = pd.read_csv(RAW / "online_retail_II.csv", dtype={"Invoice": str, "StockCode": str})
df = raw.rename(columns={"Invoice": "invoice_no", "StockCode": "stock_code", "Description": "description",
                         "Quantity": "quantity", "InvoiceDate": "invoice_datetime", "Price": "unit_price",
                         "Customer ID": "customer_id", "Country": "country"})
n_raw = len(df)
print(f"{n_raw:,} raw invoice lines")

df["stock_code"] = df["stock_code"].str.strip().str.upper()

n0 = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["invoice_no", "stock_code", "quantity", "unit_price", "customer_id", "country"])
log("Remove duplicates and double entries", n0, len(df))

n0 = len(df)
df = df[df["stock_code"].str.match(r"^\d{5}") & ~df["invoice_no"].str.startswith("A")]
log("Keep product lines only (no postage, fees, adjustments)", n0, len(df))

n0 = len(df)
df = df[df["unit_price"] > 0]
log("Remove zero-price lines (stock write-offs)", n0, len(df))

n0 = len(df)
df = df[df["quantity"].abs() <= 50000]
log("Remove two bulk-order mistakes (80,995 and 74,215 units)", n0, len(df))

df["line_type"] = np.where(df["invoice_no"].str.startswith("C"), "Return", "Sale")
df["invoice_datetime"] = pd.to_datetime(df["invoice_datetime"])
df["invoice_date"] = df["invoice_datetime"].dt.date
df["line_value"] = (df["quantity"] * df["unit_price"]).round(2)
df["customer_id"] = df["customer_id"].map(lambda x: "Unknown" if pd.isna(x) else str(int(x)))
df.groupby("line_type")["quantity"].agg(["count", "min", "max"])

1,067,371 raw invoice lines


,count,min,max
line_type,,,
Return,17912,-9360,-1
Sale,1003208,1,19152


In [3]:
# save the clean lines for the SQL part (sql/01_create_schema.sql loads this file)
order_lines = df[["invoice_no", "invoice_date", "line_type", "stock_code", "quantity", "unit_price",
                  "line_value", "customer_id", "country"]].sort_values(["invoice_date", "invoice_no"])
order_lines.to_csv(OUT / "order_lines.csv", index=False)
print(f"order_lines.csv: {len(order_lines):,} rows")

order_lines.csv: 1,021,120 rows


## 2. Weekly calendar
Planning is done in **weeks** (Monday to Sunday), because orders to suppliers and stock reviews are weekly.

- The data starts on Tuesday 01-Dec-2009, so the first **full** week starts Monday **07-Dec-2009**.
- The data stops on 09-Dec-2011. December 2011 is partial, so it is left out. The week starting 28-Nov-2011 would
  then only have 3 days (28–30 Nov), so the last full week is the one starting **21-Nov-2011**.
- That gives **103 full weeks**. Two of them (Christmas 2009 and 2010) have no sales at all: the business was closed.
  They stay in as real zero-demand weeks.

The weeks are split for testing the forecasts:

| Weeks | Dates (week starting) | Used for |
|---|---|---|
| 1–91 | 07-Dec-2009 to 29-Aug-2011 | **Train**: fit the forecast methods |
| 92–103 | 05-Sep-2011 to 21-Nov-2011 | **Test**: the 12-week backtest (the run-up to Christmas, the hardest period) |
| 104–115 | 28-Nov-2011 to 13-Feb-2012 | **Future**: the forecast used for the stock policy |
| 52–103 | 29-Nov-2010 to 21-Nov-2011 | the **last 52 weeks**: average demand, variability (CV), ABC value |

In [4]:
FIRST_WEEK = pd.Timestamp("2009-12-07")
N_WEEKS, TRAIN_WEEKS, FUTURE_WEEKS = 103, 91, 12

weeks = pd.DataFrame({"week_no": range(1, N_WEEKS + FUTURE_WEEKS + 1)})
weeks["week_start"] = FIRST_WEEK + pd.to_timedelta((weeks["week_no"] - 1) * 7, unit="D")
weeks["period"] = np.select([weeks["week_no"] <= TRAIN_WEEKS, weeks["week_no"] <= N_WEEKS], ["Train", "Test"], "Future")
weeks["in_last_52"] = weeks["week_no"].between(N_WEEKS - 51, N_WEEKS)
weeks["year"] = weeks["week_start"].dt.year
weeks["month"] = weeks["week_start"].dt.strftime("%b")
LAST_WEEK = weeks.loc[weeks.week_no == N_WEEKS, "week_start"].item()
print("history:", FIRST_WEEK.date(), "to", (LAST_WEEK + pd.Timedelta(days=6)).date())
weeks.groupby("period")["week_start"].agg(["min", "max", "count"])

history: 2009-12-07 to 2011-11-27


,min,max,count
period,,,
Future,2011-11-28,2012-02-13,12
Test,2011-09-05,2011-11-21,12
Train,2009-12-07,2011-08-29,91


In [5]:
# give every line its week (Monday of that week), then keep only lines inside the 103 full weeks
d = pd.to_datetime(df["invoice_date"])
df["week_start"] = d - pd.to_timedelta(d.dt.weekday, unit="D")
n0 = len(df)
df = df[(df["week_start"] >= FIRST_WEEK) & (df["week_start"] <= LAST_WEEK)]
log("Keep the 103 full weeks (drop 1-6 Dec 2009 and 28 Nov 2011 onwards)", n0, len(df),
    "removes the partial first week, the partial last week and the partial Dec-2011")
print(f"lines in the 103 full weeks: {len(df):,}")

lines in the 103 full weeks: 971,001


## 3. Weekly net demand per SKU
**Demand = units sold − units returned** in the week. Returns are credit notes, dated on the day the credit was raised.

Two details:
- Weeks with no sales get a **0** (a product that didn't sell still had a week of zero demand). Without this, averages and
  variability would be wrong.
- Sometimes a big return lands in a quiet week and net demand goes **negative**. A negative week isn't real demand
  (it's an earlier sale coming back), so it is set to 0. How often that happens is counted below.

In [6]:
df["sold"] = np.where(df["line_type"] == "Sale", df["quantity"], 0)
df["returned"] = np.where(df["line_type"] == "Return", -df["quantity"], 0)
df["sales_value"] = np.where(df["line_type"] == "Sale", df["line_value"], 0)
weekly_all = (df.groupby(["stock_code", "week_start"])
                .agg(units_sold=("sold", "sum"), units_returned=("returned", "sum"), sales_value=("sales_value", "sum"),
                     net_value=("line_value", "sum"))
                .reset_index())
weekly_all["net_units"] = weekly_all["units_sold"] - weekly_all["units_returned"]
print(f"{weekly_all.stock_code.nunique():,} SKUs, {len(weekly_all):,} SKU-weeks with any activity")

4,690 SKUs, 190,609 SKU-weeks with any activity


## 4. Choose the planning range: top 150 regular SKUs
A planner doesn't forecast 3,800 products one by one. We focus on the products that matter most, ranked by
**net sales value in the last 52 weeks**.

We also require a product to be **regular**: sold in at least 69 of the 103 weeks (two-thirds). One-off lines and products
launched recently don't have enough history for a 12-week backtest or a seasonal model; in a real business they would be
planned with judgement instead.

In [7]:
last52_starts = weeks.loc[weeks.in_last_52, "week_start"]
value_52 = (weekly_all[weekly_all.week_start.isin(last52_starts)]
            .groupby("stock_code")["net_value"].sum().sort_values(ascending=False))
weeks_with_sales = weekly_all[weekly_all.units_sold > 0].groupby("stock_code").size()

N_SKUS, MIN_WEEKS = 150, 69
ranking = pd.DataFrame({"net_value_52w": value_52}).join(weeks_with_sales.rename("weeks_with_sales")).fillna(0)
ranking["value_rank_all"] = np.arange(1, len(ranking) + 1)
regular = ranking[ranking.weeks_with_sales >= MIN_WEEKS]
selected = regular.head(N_SKUS).index

share = value_52[selected].sum() / value_52.sum()
print(f"regular SKUs: {len(regular):,} of {len(ranking):,}")
print(f"the top {N_SKUS} regular SKUs = {share:.1%} of all net sales in the last 52 weeks "
      f"(the 150th is number {int(ranking.loc[selected[-1], 'value_rank_all'])} overall)")

regular SKUs: 1,057 of 3,811
the top 150 regular SKUs = 34.5% of all net sales in the last 52 weeks (the 150th is number 203 overall)


In [8]:
# full grid: every selected SKU x every week, zero-filled
grid = pd.MultiIndex.from_product([selected, weeks.loc[weeks.week_no <= N_WEEKS, "week_start"]],
                                  names=["stock_code", "week_start"]).to_frame(index=False)
wd = grid.merge(weekly_all, on=["stock_code", "week_start"], how="left").fillna(
    {"units_sold": 0, "units_returned": 0, "sales_value": 0, "net_value": 0, "net_units": 0})
wd = wd.merge(weeks[["week_start", "week_no", "period"]], on="week_start")

neg = wd["net_units"] < 0
print(f"negative net weeks set to 0: {neg.sum()} of {len(wd):,} SKU-weeks ({neg.mean():.1%})")
wd["demand_units"] = wd["net_units"].clip(lower=0)
wd[["units_sold", "units_returned", "demand_units"]] = wd[["units_sold", "units_returned", "demand_units"]].astype(int)
log("Weekly grid for the 150 planning SKUs (zero weeks filled in)", len(weekly_all), len(wd),
    f"{neg.sum()} negative net weeks floored at 0")
wd.head()

negative net weeks set to 0: 107 of 15,450 SKU-weeks (0.7%)


,stock_code,week_start,units_sold,units_returned,sales_value,net_value,net_units,week_no,period,demand_units
0,22423,2009-12-07,0,0,0.00,0.00,0.00,1,Train,0
1,22423,2009-12-14,0,0,0.00,0.00,0.00,2,Train,0
2,22423,2009-12-21,0,0,0.00,0.00,0.00,3,Train,0
3,22423,2009-12-28,0,0,0.00,0.00,0.00,4,Train,0
4,22423,2010-01-04,0,0,0.00,0.00,0.00,5,Train,0


## 5. Describe each SKU
For every SKU, over the **last 52 weeks**:
- **average weekly demand** and its **standard deviation** (how much it jumps around)
- **CV** (coefficient of variation) = standard deviation ÷ average. CV 0.5 means a typical week is about 50% away from the average.
- **average selling price** = sales value ÷ units sold (sale lines only). Using net figures would break for SKU 84347:
  a credit note for 9,360 units at £0.03 (a correction, not a real return) makes its net units negative.
- **Christmas uplift** = average demand Sep–Nov 2010 (weeks 40–52) ÷ average demand over the 91 training weeks.
  1.3 or more = the SKU is **seasonal**, and Holt-Winters (which models seasonality) is allowed for it.
  The uplift uses training weeks only, so the backtest stays fair.

In [9]:
mat = wd.pivot(index="stock_code", columns="week_no", values="demand_units").loc[selected]
last52 = mat.loc[:, N_WEEKS - 51:N_WEEKS]
train = mat.loc[:, 1:TRAIN_WEEKS]

desc = (df[df.stock_code.isin(selected)].dropna(subset=["description"])
        .assign(description=lambda x: x["description"].str.strip().str.upper())
        .groupby(["stock_code", "description"]).size().reset_index(name="n")
        .sort_values("n", ascending=False).drop_duplicates("stock_code").set_index("stock_code")["description"])
sold_52 = wd[wd.week_no >= N_WEEKS - 51].groupby("stock_code")[["units_sold", "sales_value"]].sum()

sku = pd.DataFrame(index=pd.Index(selected, name="stock_code"))
sku["description"] = desc
sku["value_rank"] = np.arange(1, len(sku) + 1)
sku["net_value_52w"] = value_52[selected].round(2)
sku["units_52w"] = last52.sum(axis=1)
sku["avg_price"] = (sold_52["sales_value"] / sold_52["units_sold"]).loc[selected].round(4)
sku["avg_weekly_52w"] = last52.mean(axis=1).round(4)
sku["sd_weekly_52w"] = last52.std(axis=1, ddof=1).round(4)      # sample standard deviation, same as STDEV.S in Excel
sku["cv"] = (sku["sd_weekly_52w"] / sku["avg_weekly_52w"]).round(4)
sku["weeks_with_sales"] = ranking.loc[selected, "weeks_with_sales"].astype(int)
sku["peak_uplift"] = (mat.loc[:, 40:52].mean(axis=1) / train.mean(axis=1)).round(4)
sku["is_seasonal"] = sku["peak_uplift"] >= 1.3
sku[["avg_weekly_52w", "sd_weekly_52w", "cv", "peak_uplift"]].describe()

,avg_weekly_52w,sd_weekly_52w,cv,peak_uplift
count,150.00,150.00,150.00,150.00
mean,167.07,156.14,1.09,1.50
std,171.12,152.99,0.57,0.83
min,7.40,8.85,0.44,0.19
25%,56.05,66.54,0.68,0.97
50%,124.79,117.16,0.93,1.36
75%,210.89,176.90,1.32,1.79
max,"1,004.50","1,010.52",3.53,5.62


## 6. ABC–XYZ classification
**ABC = how much the SKU is worth** (net sales value, last 52 weeks). Sort from biggest to smallest and add up the share:
A = the SKUs that make up the first 80% of value, B = the next 15% (to 95%), C = the last 5%.

**XYZ = how predictable demand is**, using the CV of weekly demand:
X = CV ≤ 0.75 (steady), Y = 0.75 to 1.25 (variable), Z = above 1.25 (erratic).

The XYZ limits are higher than the 0.5 / 1.0 you often see in textbooks, because textbook examples use monthly
data, and weekly demand of a single product is much noisier. With 0.5 only 6 of 150 SKUs would be X.

Note: because the planning range is already the top 150 sellers, value is spread quite evenly, so A is a big group (95 SKUs).
All the limits are inputs in the Excel model, so you can try others.

In [10]:
A_LIMIT, B_LIMIT = 0.80, 0.95
X_LIMIT, Y_LIMIT = 0.75, 1.25

sku = sku.sort_values("net_value_52w", ascending=False)
sku["cum_value_share"] = (sku["net_value_52w"].cumsum() / sku["net_value_52w"].sum()).round(6)
sku["abc"] = np.select([sku.cum_value_share <= A_LIMIT, sku.cum_value_share <= B_LIMIT], ["A", "B"], "C")
sku["xyz"] = np.select([sku.cv <= X_LIMIT, sku.cv <= Y_LIMIT], ["X", "Y"], "Z")
sku["abc_xyz"] = sku["abc"] + sku["xyz"]

nine_box = pd.crosstab(sku["abc"], sku["xyz"], margins=True, margins_name="Total")
nine_box

xyz,X,Y,Z,Total
abc,,,,
A,36,31,28,95
B,10,17,12,39
C,5,7,4,16
Total,51,55,44,150


In [11]:
# share of value in each box
(pd.crosstab(sku["abc"], sku["xyz"], values=sku["net_value_52w"], aggfunc="sum", normalize=True) * 100).round(1)

xyz,X,Y,Z
abc,,,
A,33.80,25.10,20.90
B,3.80,6.50,4.50
C,1.70,2.30,1.30


## 7. Check and save

In [12]:
checks = {
    "150 SKUs, 103 weeks each": wd.groupby("stock_code").size().eq(N_WEEKS).all() and wd.stock_code.nunique() == N_SKUS,
    "no negative demand": (wd["demand_units"] >= 0).all(),
    "no missing descriptions": sku["description"].notna().all(),
    "every SKU sold in the last 52 weeks": (sku["units_52w"] > 0).all(),
    "all prices positive": (sku["avg_price"] > 0).all(),
    "classes add up": nine_box.loc["Total", "Total"] == N_SKUS,
    "no Dec-2011 dates": wd["week_start"].max() == pd.Timestamp("2011-11-21"),
}
assert all(checks.values()), checks
pd.Series(checks, name="passed").to_frame()

,passed
"150 SKUs, 103 weeks each",True
no negative demand,True
no missing descriptions,True
every SKU sold in the last 52 weeks,True
all prices positive,True
classes add up,True
no Dec-2011 dates,True


In [13]:
sku_out = sku.reset_index()[["stock_code", "description", "value_rank", "net_value_52w", "units_52w", "avg_price",
                             "avg_weekly_52w", "sd_weekly_52w", "cv", "cum_value_share", "abc", "xyz", "abc_xyz",
                             "weeks_with_sales", "peak_uplift", "is_seasonal"]]
weekly_out = wd[["stock_code", "week_no", "week_start", "period", "units_sold", "units_returned", "demand_units",
                 "sales_value", "net_value"]].sort_values(["stock_code", "week_no"])
weekly_out[["sales_value", "net_value"]] = weekly_out[["sales_value", "net_value"]].round(2)

sku_out.to_csv(OUT / "sku_list.csv", index=False)
weekly_out.to_csv(OUT / "weekly_demand.csv", index=False, date_format="%Y-%m-%d")
weeks.to_csv(OUT / "weeks.csv", index=False, date_format="%Y-%m-%d")
pd.DataFrame(prep_log).to_csv(OUT / "prep_log.csv", index=False)

for name, t in [("sku_list", sku_out), ("weekly_demand", weekly_out), ("weeks", weeks)]:
    print(f"{name:<14} {len(t):>7,} rows")
pd.DataFrame(prep_log)

sku_list           150 rows
weekly_demand   15,450 rows
weeks              115 rows


,step,rows_before,rows_after,rows_removed,note
0,Remove duplicates and double entries,1067371,1033032,34339,
1,"Keep product lines only (no postage, fees, adj...",1033032,1027052,5980,
2,Remove zero-price lines (stock write-offs),1027052,1021124,5928,
3,"Remove two bulk-order mistakes (80,995 and 74,...",1021124,1021120,4,
4,Keep the 103 full weeks (drop 1-6 Dec 2009 and...,1021120,971001,50119,"removes the partial first week, the partial la..."
5,Weekly grid for the 150 planning SKUs (zero we...,190609,15450,175159,107 negative net weeks floored at 0
